In [194]:
import numpy as np
import pandas as pd

In [195]:
import os
# os.chdir("one_hot_encoding") #change the working directory
print(os.getcwd())  # what's the current working directory
print(os.path.abspath(""))

/Users/shishir/100 days of ML/Feature Engineering/one_hot_encoding
/Users/shishir/100 days of ML/Feature Engineering/one_hot_encoding


In [196]:
df = pd.read_csv('cars.csv')
df.head()


,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [197]:
df['fuel'].value_counts() # gives you a series

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

## pd series -> df and df -> series

In [198]:
# series to dataframe
s = pd.Series([10,20,30], name='salary')
df1 = s.to_frame()
df2 = pd.DataFrame(s)

print(df1.shape, df2.shape)

# dataframe to series
s1 = df1['salary'] # selecting a column
print(type(s1))
s1


(3, 1) (3, 1)
<class 'pandas.Series'>


0    10
1    20
2    30
Name: salary, dtype: int64

## freq count using groupby

In [199]:
df.groupby('fuel').count()

,brand,km_driven,owner,selling_price
fuel,,,,
CNG,57,57,57,57
Diesel,4402,4402,4402,4402
LPG,38,38,38,38
Petrol,3631,3631,3631,3631


In [200]:
df.groupby('fuel')['fuel'].count()

fuel
CNG         57
Diesel    4402
LPG         38
Petrol    3631
Name: fuel, dtype: int64

In [201]:
df['owner'].value_counts()
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


## iloc syntax

In [202]:
df.iloc[:,-1]
# : → select all rows
# -1 → select the last column

# Returns a Series containing the values from the last column.

0       450000
1       370000
2       158000
3       225000
4       130000
         ...  
8123    320000
8124    135000
8125    382000
8126    290000
8127    290000
Name: selling_price, Length: 8128, dtype: int64

In [203]:
df.iloc[-1]
# -1 → select the last row

# Returns a Series containing all column values from the last row.

brand                   Tata
km_driven              25000
fuel                  Diesel
owner            First Owner
selling_price         290000
Name: 8127, dtype: object

In [204]:
# To keep the result as dataframe

# iloc uses
# df.iloc[row_selection, column_selection]

# select all rows and last col --> returns a series containing values of last column
print(df.iloc[:,-1])

0       450000
1       370000
2       158000
3       225000
4       130000
         ...  
8123    320000
8124    135000
8125    382000
8126    290000
8127    290000
Name: selling_price, Length: 8128, dtype: int64


In [205]:
# select all rows and last col -> returns a pandas dataframe containing a single col
df.iloc[:,[-1]].head()

,selling_price
0,450000
1,370000
2,158000
3,225000
4,130000


In [206]:
# select last row --> returns a series containing values of the last row
df.iloc[-1]

brand                   Tata
km_driven              25000
fuel                  Diesel
owner            First Owner
selling_price         290000
Name: 8127, dtype: object

In [207]:
# selects last row --> returns a pandas dataframe containing last row
df.iloc[[-1]] 

,brand,km_driven,fuel,owner,selling_price
8127,Tata,25000,Diesel,First Owner,290000


## one hot encoding using pandas and k-1 encoding

In [208]:
# drop_first=True: Drops the first category column to avoid multicollinearity. This is highly critical for linear models.
# dtype: Changes the data type of the new columns (e.g., int or float instead of the default booleans)
df_new = pd.get_dummies(data=df, columns=['fuel', 'owner'], drop_first=True, dtype=int)
df_new.head()

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0


## One Hot Encoding using sklearn

In [209]:
# 1. train test split
# 2. we will remove the fuel and owner column from X_train, apply OHE on them & then merge them back
# 3. after applying OHE, we get a CSR (sparse matrix), to convert that into numpy array use toarray()
# 4. select brand and km_driven -> this gives pd dataframe, convert them into numpy array using values()
# 5. using np.hstack() horizontally stack both the numpy arrays

# csr has 4+5-2 =  7 columns
# why -2? because we're removing first category from both features to prevent multicollinearity problem
# plus 2 cols (brand & km_driven)
# finally 9 cols

In [210]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:], df.iloc[:,-1], test_size=0.2, random_state=42)
X_train

,brand,km_driven,fuel,owner,selling_price
6518,Tata,2560,Petrol,First Owner,520000
6144,Honda,80000,Petrol,Second Owner,300000
6381,Hyundai,150000,Diesel,Fourth & Above Owner,380000
438,Maruti,120000,Diesel,Second Owner,530000
5939,Maruti,25000,Petrol,First Owner,335000
...,...,...,...,...,...
5226,Mahindra,120000,Diesel,First Owner,475000
5390,Maruti,80000,Diesel,Second Owner,530000
860,Hyundai,35000,Petrol,First Owner,576000
7603,Maruti,27000,Diesel,First Owner,770000


In [211]:
from sklearn.preprocessing import OneHotEncoder

In [212]:
enc = OneHotEncoder(drop='first', dtype='int32')

In [213]:
# fit on training data
enc.fit_transform(X_train[['fuel','owner']])

# transform on both test and training sets
X_train_enc = enc.transform(X_train[['fuel','owner']])
X_test_enc = enc.transform(X_test[['fuel','owner']])

In [214]:
X_train_enc # csr matrix

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 8718 stored elements and shape (6502, 7)>

In [215]:
# fuel has 4 categoeis + owner has 5 categories = 9
# we dropped first category from both features = 7
X_train_enc.toarray()

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 1, 0, 0]], shape=(6502, 7), dtype=int32)

## converting pd dataframe into numpy arrays

In [216]:
X_train[['brand','km_driven']].values

array([['Tata', 2560],
       ['Honda', 80000],
       ['Hyundai', 150000],
       ...,
       ['Hyundai', 35000],
       ['Maruti', 27000],
       ['Maruti', 70000]], shape=(6502, 2), dtype=object)

In [217]:
np.hstack((X_train[['brand','km_driven']].values, X_train_enc.toarray()))

array([['Tata', 2560, 0, ..., 0, 0, 0],
       ['Honda', 80000, 0, ..., 1, 0, 0],
       ['Hyundai', 150000, 1, ..., 0, 0, 0],
       ...,
       ['Hyundai', 35000, 0, ..., 0, 0, 0],
       ['Maruti', 27000, 1, ..., 0, 0, 0],
       ['Maruti', 70000, 0, ..., 1, 0, 0]], shape=(6502, 9), dtype=object)

In [218]:
enc.categories_

[array(['CNG', 'Diesel', 'LPG', 'Petrol'], dtype=object),
 array(['First Owner', 'Fourth & Above Owner', 'Second Owner',
        'Test Drive Car', 'Third Owner'], dtype=object)]

## encoding brands 32 distinct values

In [219]:
s = df['brand'].value_counts()
s

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Kia                 4
Ambassador          4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [220]:
df['brand'].nunique()

32

In [221]:
# all the brands whose freq < 100, there will come under uncommon
# for rest of the brands we will have a dedicated column
threshold = 100
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [222]:
# brands occuring >= threshold (100) times
s[s >= threshold]

brand
Maruti        2448
Hyundai       1415
Mahindra       772
Tata           734
Toyota         488
Honda          467
Ford           397
Chevrolet      230
Renault        228
Volkswagen     186
BMW            120
Skoda          105
Name: count, dtype: int64

In [179]:
# get only the brand names
frequent_brands = s[s >= threshold].index
frequent_brands

Index(['Maruti', 'Hyundai', 'Mahindra', 'Tata', 'Toyota', 'Honda', 'Ford',
       'Chevrolet', 'Renault', 'Volkswagen', 'BMW', 'Skoda'],
      dtype='str', name='brand')

In [178]:
type(s[s >= threshold].index)

pandas.Index

In [193]:
# Naive way 

# def fun(brand):
#     if brand in frequent_brands:
#         return brand
#     else:
#         return 'Others'
        
# res = []
# for brand in df['brand']: # df['brand'] series
#     res.append(fun(brand))
    
# df['brand'] = res

# df['brand'].unique()

<StringArray>
[    'Maruti',      'Skoda',      'Honda',    'Hyundai',     'Toyota',
       'Ford',    'Renault',   'Mahindra',       'Tata',  'Chevrolet',
     'Others', 'Volkswagen',        'BMW']
Length: 13, dtype: str

In [232]:
# apply() transforms the values in the series using the logic defined in the lambda
df['brand'] = df['brand'].apply(lambda x: x if x in frequent_brands else 'Others')

In [231]:
df['brand'].unique()

<StringArray>
[    'Maruti',      'Skoda',      'Honda',    'Hyundai',     'Toyota',
       'Ford',    'Renault',   'Mahindra',       'Tata',  'Chevrolet',
     'Others', 'Volkswagen',        'BMW']
Length: 13, dtype: str

In [235]:
df['brand'].isin(frequent_brands) # produces a boolean series

0       True
1       True
2       True
3       True
4       True
        ... 
8123    True
8124    True
8125    True
8126    True
8127    True
Name: brand, Length: 8128, dtype: bool

In [237]:
# use where() -- its better than apply() because it uses vectorizsed operations under the hood which are much faster for large datasets.
# apply() processes values one by one in python. It is more flexible because I can write arbitrary logic, but for simple filtering and replacement
# operations, vectorized methods like where() and isin() are preferred because they are more efficient and idiomatic in pandas

df['brand'] = df['brand'].where(
    df['brand'].isin(frequent_brands),
    'Others'
)

In [238]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [239]:
df['brand'].unique()

<StringArray>
[    'Maruti',      'Skoda',      'Honda',    'Hyundai',     'Toyota',
       'Ford',    'Renault',   'Mahindra',       'Tata',  'Chevrolet',
     'Others', 'Volkswagen',        'BMW']
Length: 13, dtype: str

In [250]:
df.head(), df.shape

(     brand  km_driven    fuel         owner  selling_price
 0   Maruti     145500  Diesel   First Owner         450000
 1    Skoda     120000  Diesel  Second Owner         370000
 2    Honda     140000  Petrol   Third Owner         158000
 3  Hyundai     127000  Diesel   First Owner         225000
 4   Maruti     120000  Petrol   First Owner         130000,
 (8128, 5))

## One hot encoding brand values

In [253]:
# avoiding the dummy variable trap: set drop_first=True
# applying ohe on brand col
brand_ohe = pd.get_dummies(data=df, columns=['brand'], dtype=int, drop_first=True)
brand_ohe

,km_driven,fuel,owner,selling_price,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Others,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen
0,145500,Diesel,First Owner,450000,0,0,0,0,0,1,0,0,0,0,0,0
1,120000,Diesel,Second Owner,370000,0,0,0,0,0,0,0,0,1,0,0,0
2,140000,Petrol,Third Owner,158000,0,0,1,0,0,0,0,0,0,0,0,0
3,127000,Diesel,First Owner,225000,0,0,0,1,0,0,0,0,0,0,0,0
4,120000,Petrol,First Owner,130000,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8123,110000,Petrol,First Owner,320000,0,0,0,1,0,0,0,0,0,0,0,0
8124,119000,Diesel,Fourth & Above Owner,135000,0,0,0,1,0,0,0,0,0,0,0,0
8125,120000,Diesel,First Owner,382000,0,0,0,0,0,1,0,0,0,0,0,0
8126,25000,Diesel,First Owner,290000,0,0,0,0,0,0,0,0,0,1,0,0


## filter rows where brand_others = 1

In [247]:
brand_ohe.where(brand_ohe['brand_Others']==1)

,km_driven,fuel,owner,selling_price,brand_BMW,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Others,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [254]:
brand_ohe['brand_Others']==1 # produces a boolean series

0       False
1       False
2       False
3       False
4       False
        ...  
8123    False
8124    False
8125    False
8126    False
8127    False
Name: brand_Others, Length: 8128, dtype: bool

In [256]:
# If you goal is to filter rows
brand_ohe[brand_ohe['brand_Others']==1]

,km_driven,fuel,owner,selling_price,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Others,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen
31,50000,Petrol,Second Owner,70000,0,0,0,0,0,0,1,0,0,0,0,0
38,42000,Petrol,First Owner,150000,0,0,0,0,0,0,1,0,0,0,0,0
41,5000,Petrol,First Owner,2100000,0,0,0,0,0,0,1,0,0,0,0,0
49,27800,Diesel,Second Owner,1450000,0,0,0,0,0,0,1,0,0,0,0,0
51,151000,Diesel,First Owner,1090000,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8072,82000,Diesel,Second Owner,450000,0,0,0,0,0,0,1,0,0,0,0,0
8090,170000,Diesel,First Owner,509999,0,0,0,0,0,0,1,0,0,0,0,0
8091,40000,Petrol,Second Owner,125000,0,0,0,0,0,0,1,0,0,0,0,0
8101,70000,Diesel,First Owner,450000,0,0,0,0,0,0,1,0,0,0,0,0
